# 03. Tool Use from Scratch

## What you'll learn

- How to define Python functions as **tools** an LLM can call
- How to describe those tools in a system prompt so the model knows what's available
- How to parse the model's response to detect tool-call requests (Action / Action Input format)
- How to dispatch calls to the right function and feed results back as **observations**
- How to wire it all into a **tool-use loop** that runs until the agent produces a final answer

## Prerequisites

- [01. Hello LLM](01_hello_llm.ipynb) — sending messages to an LLM via OpenRouter
- [02. Basic Agent Loop](02_basic_agent_loop.ipynb) — the prompt-respond-loop pattern
- [Appendix 04. Strings and JSON](../appendix/04_strings_and_json.ipynb) — regex parsing, `json.loads`, prompt templates

## Why this matters

An LLM on its own can reason about the world, but it **cannot act** in it. It can't check the weather, evaluate a math expression, or look up today's date — it can only generate text. **Tools** bridge that gap: they let the model say *"I need to call `calculate(2 + 3)`"* and have your code actually run that function and return the result. This Thought-Action-Observation cycle is the foundation of every practical agent, from simple chatbots with a calculator to complex research assistants with dozens of tools.

In this notebook we build the entire tool-use pipeline from scratch — no frameworks, just Python.

> **Further reading:**
> - [Toolformer: Language Models Can Teach Themselves to Use Tools (paper)](https://arxiv.org/abs/2302.04761) — the seminal paper on LLMs learning to invoke tools
> - [Anthropic — Tool Use (Function Calling) Docs](https://docs.anthropic.com/en/docs/build-with-claude/tool-use/overview) — production-grade tool-use patterns and best practices
> - [Lilian Weng — LLM Powered Autonomous Agents](https://lilianweng.github.io/posts/2023-06-23-agent/) — comprehensive survey of agent architectures including tool use

---

## 1. Why Tools?

Think of an LLM as a **brain** — it can reason, plan, and generate language. But it has no hands, no eyes, no internet connection. It's stuck inside a text box.

**Tools are the hands.** They let the brain reach out and interact with the world:

| What the LLM wants to do | Tool it needs |
|--------------------------|---------------|
| Do math accurately | `calculate("25 * 17 + 3")` |
| Check the weather | `get_weather("Paris")` |
| Know the current time | `get_time()` |
| Search the web | `search("latest AI news")` |
| Read a file | `read_file("data.csv")` |

Without tools, an LLM asked "what's 7,391 times 8,243?" will *guess* an answer (and probably get it wrong). With a `calculate` tool, it can get the exact result every time.

The pattern we'll build:

```
User question
    -> LLM thinks and requests a tool call
        -> Our code runs the tool
            -> Result goes back to the LLM as an "observation"
                -> LLM uses the result to answer (or calls another tool)
```

Let's build each piece.

---

## 2. Setup

First, import the `chat` helper from our shared utilities. This handles the OpenRouter API call so we can focus on the tool-use logic.

In [ ]:
import sys
# Add repo root to path so we can import from utils/
sys.path.insert(0, "..")
from utils import chat

> **Model note:** If the default model is unavailable or rate-limited, change `DEFAULT_MODEL` in `utils/__init__.py` or pass a different model to `chat()`. See [openrouter.ai/models](https://openrouter.ai/models) (filter by "Free") for current options.

In [ ]:
import re
import json
import time

---

## 3. Defining Tools as Functions

A "tool" is just a **plain Python function** that takes string arguments and returns a string result. That's it — no special decorator, no SDK, no magic.

We'll create three simple tools:
1. **`calculate`** — evaluates a math expression
2. **`get_weather`** — returns simulated weather data for a city
3. **`get_time`** — returns the current time

Each function returns a string because that's what gets injected back into the conversation as an observation.

In [ ]:
def calculate(expression: str) -> str:
    """Evaluate a math expression."""
    try:
        # Only allow safe characters — no builtins, no imports
        allowed = set("0123456789+-*/(). ")
        if not all(c in allowed for c in expression):
            return "Error: expression contains invalid characters"
        # SECURITY: eval() is used here for simplicity. In production, never use
        # eval() on untrusted input — use ast.literal_eval(), numexpr, or a
        # dedicated math parser instead. Our character whitelist is a basic
        # safeguard but not bulletproof.
        result = eval(expression)
        return str(result)
    except Exception as e:
        return f"Error: {e}"

In [ ]:
def get_weather(city: str) -> str:
    """Get weather for a city (simulated)."""
    import random
    random.seed(hash(city) % 100)  # Seed by city name for reproducible results
    temp = random.randint(15, 35)
    conditions = random.choice(["sunny", "cloudy", "rainy", "partly cloudy"])
    return f"Weather in {city}: {temp}C, {conditions}"

In [ ]:
def get_time() -> str:
    """Get the current time."""
    from datetime import datetime
    return f"Current time: {datetime.now().strftime('%H:%M:%S')}"

Let's verify each tool works on its own:

In [ ]:
print(calculate("25 * 17 + 3"))
print(get_weather("Paris"))
print(get_time())

Notice each tool:
- Takes **string** arguments (because that's what the LLM produces)
- Returns a **string** result (because that's what gets injected back into the conversation)
- Handles errors internally (returns an error message string instead of raising exceptions)

---

## 4. Describing Tools to the LLM

The LLM doesn't have access to our Python code — it only sees text. So we need to **describe** our tools in the system prompt: what each tool is called, what it does, and what arguments it takes.

This is the bridge between our Python functions and the model's understanding of what's available.

In [ ]:
TOOL_DESCRIPTIONS = """Available tools:
- calculate(expression): Evaluate a math expression. Example: calculate("2 + 3 * 4")
- get_weather(city): Get current weather for a city. Example: get_weather("Paris")
- get_time(): Get the current time. No arguments needed."""

print(TOOL_DESCRIPTIONS)

This description block will be part of the system prompt. The model reads it and learns:
- **What tools exist** (their names)
- **What each tool does** (natural language description)
- **How to call them** (argument names and example usage)

The quality of this description directly affects how well the model uses the tools. Be specific, include examples, and name the parameters clearly.

---

## 5. The Action / Action Input Format

We need to tell the model **how** to request a tool call. Free-form text like *"please use the calculator to compute 25 times 17"* is hard to parse reliably. Instead, we define a structured format:

```
Thought: <reasoning about what to do>
Action: <tool_name>
Action Input: <json arguments>
```

And for the final answer:
```
FINAL ANSWER: <the answer>
```

Why a structured format?
- **Parseable by regex** — we can reliably extract the tool name and arguments
- **Unambiguous** — the model either called a tool or it didn't
- **Separates reasoning from action** — the `Thought` field captures the model's reasoning, while `Action` and `Action Input` are the structured call

Let's build the full system prompt that includes both the tool descriptions and the format instructions:

In [ ]:
SYSTEM_PROMPT = f"""You are a helpful assistant with access to tools.

{TOOL_DESCRIPTIONS}

To use a tool, respond EXACTLY in this format:
Thought: <your reasoning>
Action: <tool_name>
Action Input: <json arguments>

When you have the final answer, respond with:
FINAL ANSWER: <your answer>

Always think before acting."""

print(SYSTEM_PROMPT)

The key phrase is **"respond EXACTLY in this format"**. LLMs are more likely to follow a format if you're explicit about it and show the exact template. Without this, the model might write something like *"I'll use the calculate tool with 25 * 17"* — valid English, but impossible to parse reliably.

---

## 6. Parsing Tool Calls

Now we need to read the model's response and figure out:
1. Did it request a tool call? (look for `Action:` and `Action Input:` lines)
2. Did it produce a final answer? (look for `FINAL ANSWER:`)
3. Or did it just respond with plain text?

We'll use regex to extract the structured parts. If you need a refresher on regex, see [Appendix 04](../appendix/04_strings_and_json.ipynb).

In [ ]:
def parse_action(response):
    """Parse Action/Action Input from LLM response.
    
    Returns a dict with one of three types:
      - {"type": "tool_call", "tool": ..., "input": ...}
      - {"type": "final_answer", "answer": ...}
      - {"type": "text", "content": ...}
    """
    # re.MULTILINE makes ^ and $ match start/end of each line, not just the whole string
    action_match = re.search(r"^Action:\s*(.+)$", response, re.MULTILINE)
    input_match = re.search(r"^Action Input:\s*(.+)$", response, re.MULTILINE)

    if action_match:
        tool_name = action_match.group(1).strip()
        tool_input = {}
        if input_match:
            raw = input_match.group(1).strip()
            try:
                tool_input = json.loads(raw)
            except json.JSONDecodeError:
                # If it's not valid JSON, wrap it as a raw string
                tool_input = {"raw": raw}
        return {"type": "tool_call", "tool": tool_name, "input": tool_input}

    # Look for FINAL ANSWER:
    if "FINAL ANSWER:" in response:
        answer = response.split("FINAL ANSWER:")[-1].strip()
        return {"type": "final_answer", "answer": answer}

    # Neither — just plain text
    return {"type": "text", "content": response}

Let's test the parser with several example responses to make sure it handles each case:

In [ ]:
# Test 1: A proper tool call
test_tool_call = """Thought: I need to calculate this.
Action: calculate
Action Input: {"expression": "25 * 17"}"""

result = parse_action(test_tool_call)
print("Test 1 (tool call):")
print(f"  Type: {result['type']}")
print(f"  Tool: {result['tool']}")
print(f"  Input: {result['input']}")
print()

In [ ]:
# Test 2: A final answer
test_final = "FINAL ANSWER: The weather in Paris is 22C and sunny."

result = parse_action(test_final)
print("Test 2 (final answer):")
print(f"  Type: {result['type']}")
print(f"  Answer: {result['answer']}")
print()

In [ ]:
# Test 3: A tool call with no arguments (like get_time)
test_no_args = """Thought: The user wants to know the time.
Action: get_time
Action Input: {}"""

result = parse_action(test_no_args)
print("Test 3 (no-arg tool):")
print(f"  Type: {result['type']}")
print(f"  Tool: {result['tool']}")
print(f"  Input: {result['input']}")
print()

In [ ]:
# Test 4: Plain text (no action, no final answer)
test_plain = "I'm not sure what you mean. Could you clarify?"

result = parse_action(test_plain)
print("Test 4 (plain text):")
print(f"  Type: {result['type']}")
print(f"  Content: {result['content']}")

The parser handles all three cases cleanly. Note the fallback behavior: if `Action Input` isn't valid JSON, we wrap it in `{"raw": ...}` so the dispatch step can still work with it.

---

## 7. Dispatching Tool Calls

Once we've parsed the model's response and know *which* tool it wants and *what arguments* to pass, we need to actually call the function. This is **dispatch** — looking up the tool by name and invoking it.

The key data structure is the **tool registry**: a dictionary mapping tool names (strings) to Python functions.

In [ ]:
tool_registry = {
    "calculate": calculate,
    "get_weather": get_weather,
    "get_time": get_time,
}

print(f"Registered tools: {list(tool_registry.keys())}")

In [ ]:
def dispatch_tool(tool_name, tool_input):
    """Look up a tool by name and call it with the given arguments.
    
    Returns the tool's string result, or an error message if something goes wrong.
    """
    if tool_name not in tool_registry:
        return f"Error: Unknown tool '{tool_name}'. Available: {list(tool_registry.keys())}"
    
    fn = tool_registry[tool_name]
    try:
        return fn(**tool_input)
    except TypeError as e:
        return f"Error calling {tool_name}: {e}"

Let's test the dispatch with each tool:

In [ ]:
# Dispatch to calculate
print(dispatch_tool("calculate", {"expression": "100 / 4 + 7"}))

# Dispatch to get_weather
print(dispatch_tool("get_weather", {"city": "Tokyo"}))

# Dispatch to get_time (no arguments)
print(dispatch_tool("get_time", {}))

# Dispatch to unknown tool — should give a helpful error
print(dispatch_tool("search", {"query": "test"}))

# Dispatch with wrong arguments — should give a helpful error
print(dispatch_tool("calculate", {"wrong_arg": "2 + 2"}))

Notice how errors are returned as strings rather than raised as exceptions. This is important: when a tool call fails, we want the LLM to see the error message and potentially **retry** with corrected arguments, rather than crashing the whole loop.

---

## 8. The Observation Step

After a tool runs and returns a result, we need to feed that result **back to the LLM** so it can decide what to do next. This is the "observation" step — the model observes the result of its action.

We inject the tool result as a **user message** in this format:

```
Observation: <tool result>

Continue reasoning. If you have the final answer, respond with FINAL ANSWER: <answer>
```

Why a user message? Because in the chat format, the model can only see `system`, `user`, and `assistant` messages. Tool results aren't a built-in role in the basic chat API, so we use the `user` role as the carrier.

The second line — *"Continue reasoning..."* — is a gentle nudge reminding the model of the format. Without it, the model sometimes forgets to use `FINAL ANSWER:` and just responds in free text.

In [ ]:
# Example: what an observation message looks like
tool_result = dispatch_tool("calculate", {"expression": "25 * 17"})

observation = f"Observation: {tool_result}\n\nContinue reasoning. If you have the final answer, respond with FINAL ANSWER: <answer>"

print(observation)

This observation message gets appended to the conversation as `{"role": "user", "content": observation}`. The LLM then sees the full conversation so far — including its own tool request and the result — and can decide whether to call another tool or deliver the final answer.

---

## 9. The Complete Tool-Use Loop

Now we combine everything into a single function that:

1. Sends the user's question to the LLM (with the system prompt describing tools)
2. Parses the response — is it a tool call, a final answer, or plain text?
3. If it's a tool call: dispatches the tool, injects the observation, and loops back to step 1
4. If it's a final answer: returns it
5. Safety limit: stops after `max_steps` iterations to prevent infinite loops

This is the **Thought-Action-Observation** cycle — the core of tool-using agents.

In [ ]:
def tool_agent(user_query, max_steps=10):
    """Run a tool-using agent loop.
    
    Args:
        user_query: The user's question.
        max_steps: Safety limit on LLM calls to prevent infinite loops.
    
    Returns:
        The final answer string, or None if max_steps is exceeded.
    """
    system_prompt = f"""You are a helpful assistant with access to tools.

{TOOL_DESCRIPTIONS}

To use a tool, respond EXACTLY in this format:
Thought: <your reasoning>
Action: <tool_name>
Action Input: <json arguments>

When you have the final answer, respond with:
FINAL ANSWER: <your answer>

Always think before acting."""

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_query},
    ]

    for step in range(1, max_steps + 1):
        try:
            response = chat(messages)
        except RuntimeError as e:
            print(f"\nAPI error: {e}")
            return None

        print(f"\n--- Step {step} ---")
        print(f"Assistant: {response}")

        messages.append({"role": "assistant", "content": response})

        parsed = parse_action(response)

        if parsed["type"] == "final_answer":
            print(f"\n=== Final Answer: {parsed['answer']} ===")
            return parsed["answer"]

        elif parsed["type"] == "tool_call":
            result = dispatch_tool(parsed["tool"], parsed["input"])
            print(f"Tool [{parsed['tool']}]: {result}")
            observation = (
                f"Observation: {result}\n\n"
                f"Continue reasoning. If you have the final answer, "
                f"respond with FINAL ANSWER: <answer>"
            )
            messages.append({"role": "user", "content": observation})

        else:
            # Plain text — nudge the model to use the format
            messages.append({
                "role": "user",
                "content": "Please use a tool or provide a FINAL ANSWER.",
            })

        time.sleep(1)  # Rate-limit between LLM calls

    print(f"\nMax steps ({max_steps}) reached without final answer.")
    return None

Let's break down what's happening in the loop:

1. **`chat(messages)`** — send the full conversation to the LLM and get a response
2. **`parse_action(response)`** — determine if the response is a tool call, final answer, or text
3. **Branch on type:**
   - `final_answer` — we're done, return it
   - `tool_call` — run the tool, inject the result as an observation, loop again
   - `text` — the model didn't follow the format, nudge it to try again
4. **`time.sleep(1)`** — basic rate limiting to be respectful to the API
5. **`max_steps` safety limit** — prevents infinite loops if the model never produces a final answer

### Test 1: A simple single-tool question

Let's start with something straightforward — asking for the current time. This should require exactly one tool call.

In [ ]:
answer = tool_agent("What time is it right now?")

You should see the agent:
1. Think about needing to check the time
2. Call `get_time` with no arguments
3. See the observation with the current time
4. Produce a `FINAL ANSWER` with the time

That's the full Thought-Action-Observation cycle in action!

---

## 10. Putting It Together: Multi-Tool Query

The real power of tool-use agents shows up when a question requires **multiple tools in sequence**. The model needs to:
1. Call the first tool and read the result
2. Use information from that result to decide what to do next
3. Call a second tool
4. Combine everything into a final answer

Let's try a question that requires both `get_weather` and `calculate`:

In [ ]:
answer = tool_agent(
    "What's the weather in Paris? And what's 25 times the temperature you find?"
)

Watch the trace above. The agent should:

1. **Step 1:** Call `get_weather("Paris")` to find the temperature
2. **Step 2:** See the observation (e.g., "Weather in Paris: 22C, sunny"), extract the temperature, then call `calculate("25 * 22")`
3. **Step 3:** See the calculation result and combine everything into a `FINAL ANSWER`

This is **chained tool use** — the output of one tool feeds into the input of the next. The LLM is doing the reasoning about *what to do next* based on *what it observed*. Our code just handles the plumbing.

Let's try one more — a math-only question to confirm the calculator works end-to-end:

In [ ]:
answer = tool_agent("What is (145 + 67) * 3 - 18?")

---

## Try It Yourself

Work through these exercises to deepen your understanding. Each one extends the tool-use agent you just built.

### Exercise 1: Add a `search` tool

Create a `search(query)` tool that returns simulated search results. Then:
1. Add it to the `tool_registry`
2. Update `TOOL_DESCRIPTIONS` to include it
3. Run the agent with a question like *"Search for the tallest building in the world"*

```python
def search(query: str) -> str:
    """Search the web (simulated)."""
    # Return 2-3 fake but plausible results
    # Hint: you can hardcode results or use random.choice for variety
    pass
```

In [ ]:
# Exercise 1: Your code here


### Exercise 2: Handle unknown tools gracefully

Our `dispatch_tool` already returns an error for unknown tools. But does the **agent** handle it well? Test it:

1. Temporarily remove `get_weather` from the registry (but leave it in the system prompt)
2. Run the agent with a weather question
3. Does it see the error and retry, or does it get stuck?

If needed, improve the observation message to help the model recover.

In [ ]:
# Exercise 2: Your code here


### Exercise 3: Add tool call counting

Modify the `tool_agent` function to track how many times each tool is called. After the loop finishes (whether by final answer or max steps), print a summary like:

```
Tool usage summary:
  get_weather: 1 call(s)
  calculate: 1 call(s)
  Total steps: 3
```

Hint: use a `collections.Counter` or a plain dict.

In [ ]:
# Exercise 3: Your code here


### Exercise 4: Enforce reasoning before action

The system prompt asks the model to include a `Thought:` line before each `Action:`. But sometimes models skip the thought.

Modify `parse_action` to check whether the response contains a `Thought:` line when it also contains an `Action:`. If the thought is missing, return a special type like `{"type": "missing_thought"}` and have the agent loop send a reminder message.

Hint: add a `re.search(r"^Thought:\s*(.+)$", response, re.MULTILINE)` check.

In [ ]:
# Exercise 4: Your code here


---

## Summary

In this notebook you built a complete tool-use agent from scratch. Here's what each piece does:

| Component | What it does |
|-----------|-------------|
| **Tool functions** | Plain Python functions that take strings and return strings |
| **Tool descriptions** | Text in the system prompt telling the LLM what tools exist |
| **Action format** | A structured text format (`Action:` / `Action Input:`) the LLM uses to request tool calls |
| **Parser** | Regex-based extraction of tool name and arguments from the LLM's response |
| **Registry + dispatch** | A dict mapping tool names to functions, plus the code to call them |
| **Observation** | Feeding the tool result back to the LLM as a user message |
| **Loop** | The while-loop that repeats until the LLM produces a final answer |

This is the same architecture used by production agents — the difference is that frameworks like smolagents, LangChain, and others automate the parsing, dispatch, and observation injection. Now you know exactly what's happening under the hood.

Next up: [04. Structured Output](04_structured_output.ipynb) — getting the LLM to return clean JSON instead of free text, which makes parsing much more reliable.

### From teaching to production

This notebook built the tool-use pipeline with minimal code to focus on the mechanics. In a production tool-use agent, you'd also want:

- **Argument validation against schemas** — verify tool arguments match expected types and ranges before executing
- **Sandboxed execution** — run tool code in an isolated environment to prevent side effects from malformed inputs
- **Tool-call timeouts** — cap execution time per tool call to prevent hanging on slow or stuck tools